In [ ]:
import os
import torch
from dotenv import load_dotenv

load_dotenv()
access_token = os.environ.get("HF_TOKEN")
if access_token is None:
    raise ValueError("Set HF_TOKEN in your environment or in a local .env file.")

if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='mps')

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-3b")

config = {
    "tokenizer": tokenizer,
    "batch_size": 4,
    "context_length": 512,
    "vocab_size": len(tokenizer),
    "embedding_dim": 768,
    "n_layers": 8, 
    "n_heads": 12
}

batch_size = config["batch_size"]
context_length = config["context_length"]
vocab_size = config["vocab_size"]
embedding_dim = config["embedding_dim"]
n_heads = config["n_heads"]
n_layers = config["n_layers"]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


In [4]:
from data import create_dataset, CoderDataset
from torch.utils.data import DataLoader

small_dataset = create_dataset(access_token).take(20)
coder_dataset = CoderDataset(
    dataset=small_dataset,
    tokenizer=tokenizer,
    context_length=context_length
)

data_loader = DataLoader(
    dataset=coder_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

x_batch, y_batch = next(iter(data_loader))
print(f"x batch shape is:", x_batch.shape)
print(f"y batch shape is:", y_batch.shape)

x batch shape is: torch.Size([4, 512])
y batch shape is: torch.Size([4, 512])


In [5]:
token_embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)
x_embedded = token_embedding_layer(x_batch)
x_embedded.shape

torch.Size([4, 512, 768])

In [6]:
q_proj_layer = torch.nn.Linear(embedding_dim, embedding_dim)
k_proj_layer = torch.nn.Linear(embedding_dim, embedding_dim)
v_proj_layer = torch.nn.Linear(embedding_dim, embedding_dim)

In [7]:
query_matrix = q_proj_layer(x_embedded)
key_matrix = k_proj_layer(x_embedded)
value_matrix = v_proj_layer(x_embedded)

query_matrix.shape

torch.Size([4, 512, 768])

In [8]:
if embedding_dim % n_heads == 0:
    head_d = embedding_dim // n_heads
    query_heads = query_matrix.reshape(batch_size, context_length, n_heads, head_d)
    key_heads = key_matrix.reshape(batch_size, context_length, n_heads, head_d)
    value_heads = value_matrix.reshape(batch_size, context_length, n_heads, head_d)

query_heads.shape

torch.Size([4, 512, 12, 64])

In [9]:
query_heads = query_heads.transpose(1, 2)
key_heads = key_heads.transpose(1, 2)
value_heads = value_heads.transpose(1, 2)

query_heads.shape

torch.Size([4, 12, 512, 64])

In [ ]:
from torch import nn

class Attention(nn.Module):
    def __init__(self, embedding_dim, n_heads, n_kv_heads, dropout=0.0, is_causal=True):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.head_dim = embedding_dim // n_heads
        self.dropout = dropout
        self.is_causal = is_causal

        self.query_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.key_proj = nn.Linear(embedding_dim, n_kv_heads * self.head_dim, bias=False)
        self.value_proj = nn.Linear(embedding_dim, n_kv_heads * self.head_dim, bias=False)

        self.output_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)

    def forward(self, x):
        # We take input with shape (batch_size, context_length, embedding_dim)..
        # We want to project query, key, and value matrices from the input
        # We then reshape and transpose them to have shape (batch_size, n_head, context_length, head_dim)
        # We calculate attention with each of those heads
        # Then we merge the heads back to shape (batch_size, context_length, embedding_dim) and return that

        batch_size, context_length, embedding_dim = x.shape
        query_matrix = self.query_proj(x)
        key_matrix = self.key_proj(x)
        value_matrix = self.value_proj(x)

        query_heads = query_matrix.reshape(batch_size, context_length, self.n_heads, self.head_dim).transpose(1, 2)
        key_heads = key_matrix.reshape(batch_size, context_length, self.n_kv_heads, self.head_dim).transpose(1, 2)
        value_heads = value_matrix.reshape(batch_size, context_length, self.n_kv_heads, self.head_dim).transpose(1, 2)

        key_heads = key_heads.repeat_interleave(self.n_rep, dim=1)
        value_heads = value_heads.repeat_interleave(self.n_rep, dim=1)

        grouped_query_attention = nn.functional.scaled_dot_product_attention(
            query_heads,
            key_heads,
            value_heads,
            dropout_p = self.dropout if self.training else 0.0,
            is_causal=self.is_causal
        )

        merged_attention = grouped_query_attention.transpose(1, 2).flatten(start_dim=2)
        return self.output_proj(merged_attention)






In [62]:
mh_attention_output = torch.nn.functional.scaled_dot_product_attention(
    query_heads,
    key_heads,
    value_heads,
    dropout_p=0.0,
    is_causal=True
)

mh_attention_output.shape

torch.Size([4, 12, 512, 64])

In [64]:
merged_attention = mh_attention_output.transpose(1, 2).flatten(start_dim=2)
merged_attention.shape

torch.Size([4, 512, 768])

In [ ]:
out_proj_layer = torch.nn.Linear(embedding_dim, embedding_dim)
output_projection = out_proj_layer(merged_attention)

output_projection.shape

torch.Size([4, 512, 768])